# Project EXPEDIA: Sequence Embedding

This notebook embeds genomic sequences using Nucleotide Transformer v2-50M on an A100 or T4 GPU.

In [ ]:
!pip install transformers torch einops polars pyarrow

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import polars as pl
from tqdm.auto import tqdm
import gc

# 1. Model Loading and Patching
model_name = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Apply intermediate_size patch for SwiGLU
model = AutoModelForMaskedLM.from_pretrained(
    model_name,
    trust_remote_code=True,
).cuda()

# Force monkey-patch intermediate_size
# As per instructions, implement the intermediate_size=4096 monkey-patch for SwiGLU.
for layer in model.esm.encoder.layer:
    if hasattr(layer.intermediate, 'dense'):
        layer.intermediate.dense.out_features = 4096

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model patched and loaded on {device}")

In [ ]:
# 2. Processing Loop
def kmer_tokenize(sequence, k=6):
    """6-mer sliding window tokenization"""
    # Overlapping 6-mers
    kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
    return " ".join(kmers)

def process_chunk(parquet_path, output_path, batch_size=64):
    print(f"Processing {parquet_path}")
    df = pl.read_parquet(parquet_path)
    
    all_embeddings = []
    accessions = df["accession"].to_list()
    sequences = df["sequence"].to_list()
    taxonomy_ids = df["taxonomy_id"].to_list()

    # Memory Optimization: Mixed Precision and dynamic batches
    scaler = torch.cuda.amp.autocast()
    
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i:i+batch_size]
        kmer_seqs = [kmer_tokenize(seq) for seq in batch_seqs]
        
        # Tokenize
        inputs = tokenizer(kmer_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            with scaler: # torch.amp mixed precision
                outputs = model(**inputs, output_hidden_states=True)
                # Get last hidden state
                hidden_states = outputs.hidden_states[-1] # shape (batch_size, seq_len, 512)
                
                # Mean pooling for sequence representation
                mask = inputs["attention_mask"].unsqueeze(-1).expand(hidden_states.size()).float()
                sum_embeddings = torch.sum(hidden_states * mask, 1)
                sum_mask = mask.sum(1)
                mean_pooled = sum_embeddings / torch.clamp(sum_mask, min=1e-9)
                
        # Generate 512D embeddings and zero-pad to 768D
        pad_size = 768 - mean_pooled.size(1)
        padded_embeddings = torch.nn.functional.pad(mean_pooled, (0, pad_size))
        
        all_embeddings.extend(padded_embeddings.cpu().numpy().tolist())
        
        del inputs, outputs, hidden_states, mean_pooled, padded_embeddings
        torch.cuda.empty_cache()
        
    # Save Output
    out_df = pl.DataFrame({
        "accession": accessions,
        "taxonomy_id": taxonomy_ids,
        "vector": all_embeddings
    })
    
    # List of floats vector column natively supported by pyarrow/polars parquet
    out_df.write_parquet(output_path)
    print(f"Saved {output_path}")


In [ ]:
# Example usage:
# process_chunk("raw_sequences_001.parquet", "embedded_sequences_001.parquet", batch_size=64)